In [18]:
import os
import re
import uuid
from typing import List, Dict

import fitz  # PyMuPDF
import pdfplumber
from tqdm import tqdm

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

In [2]:
def detect_section_heading(text: str) -> str:
    """
    Detect possible section headings.

    Many rule books contain headings like:
    1. Introduction
    2.3 Compliance Rules
    Rule 4: Eligibility

    This function tries to detect them.
    """

    lines = text.split("\n")

    for line in lines[:5]:
        line = line.strip()

        if len(line) < 120 and re.match(r"^(\d+(\.\d+)*)\s+.+", line):
            return line

        if line.lower().startswith("rule"):
            return line

    return "unknown_section"


In [3]:
def extract_text(pdf_path: str) -> List[Document]:
    """
    Extract raw text from PDF with page metadata.
    """

    documents = []

    doc = fitz.open(pdf_path)

    for page_num in tqdm(range(len(doc)), desc="Extracting text"):
        page = doc.load_page(page_num)

        text = page.get_text()

        if not text.strip():
            continue

        section = detect_section_heading(text)

        documents.append(
            Document(
                page_content=text,
                metadata={
                    "page": page_num + 1,
                    "chunk_type": "text",
                    "section": section,
                    "source": "rule_book"
                }
            )
        )

    return documents

In [4]:
def extract_tables(pdf_path: str) -> List[Document]:

    table_docs = []

    with pdfplumber.open(pdf_path) as pdf:

        for page_num, page in enumerate(tqdm(pdf.pages, desc="Extracting tables")):

            tables = page.extract_tables()

            for table in tables:

                if not table:
                    continue

                header = [str(c) if c is not None else "" for c in table[0]]
                rows = [[str(c) if c is not None else "" for c in row] for row in table[1:]]

                md = "| " + " | ".join(header) + " |\n"
                md += "| " + " | ".join(["---"] * len(header)) + " |\n"

                for row in rows:
                    md += "| " + " | ".join(row) + " |\n"

                table_docs.append(
                    Document(
                        page_content=md,
                        metadata={
                            "page": page_num + 1,
                            "chunk_type": "table",
                            "section": "table_data",
                            "source": "rule_book"
                        }
                    )
                )

    return table_docs

In [5]:
PDF_PATH = "./pdf/tata_ipl_2024.pdf"
CHROMA_PATH = "./chroma_db"

# Chunking parameters tuned for rule documents
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

text_docs = extract_text(PDF_PATH)
print(f"\nText pages extracted: {len(text_docs)}")

Extracting text: 100%|██████████| 36/36 [00:00<00:00, 307.24it/s]


Text pages extracted: 36


In [6]:
table_docs = extract_tables(PDF_PATH)
print(f"Tables extracted: {len(table_docs)}")
# print sample table content
if table_docs:
    print("\nSample table content (Markdown format):")
    print(table_docs[0].page_content)

Extracting tables: 100%|██████████| 36/36 [00:02<00:00, 14.10it/s]

Tables extracted: 24

Sample table content (Markdown format):
|  | 2.1 |  |  | Excessive appealing during a Match |  |
| --- | --- | --- | --- | --- | --- |
| Note: |  |  | For the purpose of Article 2.1, ‘excessive’ may include (a) repeated appealing of the same decision;
(b) repeated appealing of different decisions when the bowler/fielder knows the batter is not out
with the intention of placing the Umpire under pressure; (c) charging or advancing towards the
Umpire in an aggressive manner when appealing; or (d) celebrating a dismissal without appealing
to the Umpire when a decision is required. It is not intended to prevent loud or enthusiastic
appealing. |  |  |
| Level 1 |  |  | ü |  |  |
| Level 2 |  |  | Not applicable |  |  |
| Level 3 |  |  | Not applicable |  |  |
| Level 4 |  |  | Not applicable |  |  |
|  |  |  |  |  |  |
| 2.2 |  |  |  | Abuse of cricket equipment or clothing, ground equipment or fixtures and fittings |  |
|  |  |  |  | during a Match. |  |
| Note: |  |  

In [7]:
def chunk_text_documents(docs: List[Document]) -> List[Document]:
    """
    Split long text into semantic chunks.

    Tables are not passed here (handled separately).
    """

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=[
            "\n\n",
            "\n",
            ". ",
            " "
        ]
    )

    chunks = []

    for doc in docs:

        splits = splitter.split_text(doc.page_content)

        for chunk in splits:

            chunks.append(
                Document(
                    page_content=chunk,
                    metadata=doc.metadata
                )
            )

    return chunks

In [8]:
# Chunk text
text_chunks = chunk_text_documents(text_docs)

print(f"\nText chunks created: {len(text_chunks)}")


Text chunks created: 135


In [9]:
# Combine
all_docs = text_chunks + table_docs

print(f"\nTotal chunks to embed: {len(all_docs)}")


Total chunks to embed: 159


In [10]:
def load_embedding_model():
    """
    Load high quality embedding model.

    Model: all-mpnet-base-v2
    Reason:
    - Strong semantic retrieval
    - Excellent for long documents
    - Open source
    """

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2"
    )

    return embeddings

In [11]:
# ----------------------------------------
# Store vectors in ChromaDB
# ----------------------------------------

def store_in_chroma(documents, embedding_model):

    os.makedirs(CHROMA_PATH, exist_ok=True)

    # Filter empty documents
    clean_docs = [
        doc for doc in documents
        if doc.page_content and doc.page_content.strip()
    ]

    if len(clean_docs) == 0:
        raise ValueError("No valid documents found after filtering empty chunks.")

    print(f"Storing {len(clean_docs)} valid chunks in ChromaDB")

    ids = [str(uuid.uuid4()) for _ in clean_docs]

    vectordb = Chroma.from_documents(
        documents=clean_docs,
        embedding=embedding_model,
        persist_directory=CHROMA_PATH,
        ids=ids
    )

    # vectordb.persist()

    return vectordb


In [12]:
# Load embedding model
embeddings = load_embedding_model()

/var/folders/8q/yv4mb1zs2j7ddg7j76tc8skh0000gn/T/ipykernel_77901/3921473616.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1946.64it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
# Store in Chroma
vectordb = store_in_chroma(all_docs, embeddings)
print("\nVector database created successfully.")

Storing 159 valid chunks in ChromaDB

Vector database created successfully.


In [14]:
def test_retrieval(vectordb):
    """
    Simple verification query.
    """

    query = "What is period of suspension?"

    results = vectordb.similarity_search(query, k=5)

    print("\n=============================")
    print("RETRIEVAL TEST RESULTS")
    print("=============================\n")

    for i, doc in enumerate(results):

        print(f"Result {i+1}")
        print(f"Page:", doc.metadata["page"])
        print(f"Type:", doc.metadata["chunk_type"])
        print(f"Section:", doc.metadata["section"])
        print("-" * 50)
        print(doc.page_content[:1000])
        print("\n")


In [15]:
# Run retrieval test
test_retrieval(vectordb)


RETRIEVAL TEST RESULTS

Result 1
Page: 25
Type: text
Section: unknown_section
--------------------------------------------------
7.7 
Period of Suspension 
 
7.7.1 
In imposing any suspension of any number of Matches on a Player or Team Official the 
Match Referee or Ombudsman shall select the Matches which are the most proximate (i.e. 
nearest in time) to the Suspension Date.  However, where selecting the most proximate 
Match(es) would result in the suspension being applied in relation to a Match in which 
the Player would not participate or would, at the Suspension Date, not be likely to 
participate in each case as a result of the Player being involved in International Duty then 
the Match Referee or Ombudsman shall in imposing the suspension choose the next most 
proximate Match(es).


Result 2
Page: 25
Type: text
Section: unknown_section
--------------------------------------------------
7.7 
Period of Suspension 
 
7.7.1 
In imposing any suspension of any number of Matches on a

In [17]:
#write a gradio app to query this vector database
import gradio as gr
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings    

def query_vectordb(query: str) -> str:

    if not query.strip():
        return "Please enter a valid query."

    results = vectordb.similarity_search(query, k=5)

    if not results:
        return "No relevant information found."

    response = "Top relevant chunks:\n\n"

    for i, doc in enumerate(results):
        response += f"Result {i+1}\n"
        response += f"Page: {doc.metadata['page']}\n"
        response += f"Type: {doc.metadata['chunk_type']}\n"
        response += f"Section: {doc.metadata['section']}\n"
        response += "-" * 50 + "\n"
        response += doc.page_content[:1000] + "\n\n"

    return response 

with gr.Blocks() as demo:
    gr.Markdown("# Rule Book Query App")
    gr.Markdown("Ask questions about the rule book and get relevant sections.")

    query_input = gr.Textbox(label="Enter your query here", lines=2)
    submit_btn = gr.Button("Search")
    output_area = gr.Textbox(label="Results", lines=20)

    submit_btn.click(fn=query_vectordb, inputs=query_input, outputs=output_area)
demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
